# Simple CNN testing

In [1]:
import torch
import wandb
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
# Check for GPU
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")

print(device)

mps


In [3]:
transformer = transforms.Compose([
    transforms.Resize((224, 224)), # transform to size = 224x224
    transforms.ToTensor(), # transforms into tensor
])

full_dataset = datasets.ImageFolder("../data/icosimal_img_class_03/train", transform=transformer)
splits = torch.load("../data/split/split_train_test_indices.pth")

train_dataset = torch.utils.data.Subset(full_dataset, splits['train_idx'])
test_dataset = torch.utils.data.Subset(full_dataset, splits['test_idx'])
val_dataset = datasets.ImageFolder("../data/icosimal_img_class_03/validate", transform=transformer)

In [4]:
# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Training dataset size: 20000
Validation dataset size: 6000
Test dataset size: 4000


### Create Training loop for models

In [5]:
def train_eval(model, optimizer, nepochs, batch_size, training_data, validation_data, device, project='MPW-CNN',run_name=None, use_wandb=True):
    """
    Train and evaluate a model.
    Logs train/validation loss and accuracy to Weights & Biases if use_wandb=True.
    """
    cost_hist = []
    cost_hist_test = []
    acc_hist = []
    acc_hist_test = []

    model = model.to(device) # <-- move model to device (GPU or CPU)
    cost_ce = torch.nn.CrossEntropyLoss().to(device)

    train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=False)

    if use_wandb:
        wandb.init(
            project=project,
            name=run_name,
            config={
                "nepochs": nepochs,
                "batch_size": batch_size,
                "optimizer": optimizer.__class__.__name__,
                "loss": "CrossEntropyLoss",
                "device": str(device),
                "model": model.__class__.__name__
            }
        )
        wandb.watch(model, log="all", log_freq=100)

    for epoch in range(nepochs):
        model.train()
        size = len(train_loader.dataset)
        nbatches = len(train_loader)
        cost, acc = 0.0, 0.0
        for batch, (X, Y) in enumerate(train_loader):
            X,Y = X.to(device),Y.to(device)
            pred = model(X)
            loss = cost_ce(pred, Y)
            cost += loss.item()
            acc += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

            # gradient, parameter update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        cost /= nbatches
        acc /= size

        model.eval()
        size_test = len(val_loader.dataset)
        nbatches_test = len(val_loader)
        cost_test, acc_test = 0.0, 0.0

        with torch.no_grad():
            for X, Y in val_loader:
                X,Y = X.to(device),Y.to(device)
                pred = model(X)
                cost_test += cost_ce(pred, Y).item()
                acc_test += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

        cost_test /= nbatches_test
        acc_test /= size_test

        print("Epoch %i: %f, %f, %f, %f"%(epoch, cost, acc, cost_test, acc_test))

        cost_hist.append(cost)
        cost_hist_test.append(cost_test)
        acc_hist.append(acc)
        acc_hist_test.append(acc_test)

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": cost,
                "train_accuracy": acc,
                "val_loss": cost_test,
                "val_accuracy": acc_test,
                "lr": optimizer.param_groups[0]['lr']
            })

    if use_wandb:
        wandb.finish()

    return cost_hist, cost_hist_test, acc_hist, acc_hist_test

### Creating simple CNN-model

In [6]:
# creat a simple model with one convolutional layer and two fully connected layers

class simple_model(nn.Module):

    def __init__(self, units=100):
        super(simple_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )


    def forward(self, x):
        return self.seq(x)

In [7]:
# create an model and its summary
# model = simple_model(100).to(device)
from torchsummary import summary

# this is added because of mps device in Apple products
# this way summary() works as it should
model_cpu = simple_model(100).cpu()
summary(model_cpu, (3,224,224), device='cpu')
model = model_cpu.to(device)


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 100]      40,140,900
              ReLU-6                  [-1, 100]               0
            Linear-7                   [-1, 10]           1,010
Total params: 40,142,806
Trainable params: 40,142,806
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 153.13
Estimated Total Size (MB): 184.33
----------------------------------------------------------------


Initiate Training

In [7]:
batch_size = 32
nepochs = 10
lr = 0.1
units = 100

model = simple_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model,
                                                                          optimizer, nepochs, batch_size,
                                                                          train_dataset, val_dataset,
                                                                          device, project='MPW-CNN',
                                                                          run_name='simple_model',
                                                                          use_wandb=True)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/jdemid/.netrc.
wandb: Currently logged in as: jdem (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run phrwun7j
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /Users/jdemid/Projects/PycharmProjects/deep-learning-mpw/01_MPW-CNN/notebooks/wandb/run-20260325_111057-phrwun7j
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run simple_model
wandb: ⭐️ View project at https://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN
wandb: 🚀 View run at https://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN/runs/phrwun7j


Epoch 0: 2.280445, 0.154800, 2.193506, 0.198833
Epoch 1: 2.134323, 0.229950, 2.078785, 0.255667
Epoch 2: 2.023581, 0.281150, 2.024151, 0.286167
Epoch 3: 1.910709, 0.322800, 1.938289, 0.312000
Epoch 4: 1.756268, 0.387100, 1.944116, 0.308667
Epoch 5: 1.530577, 0.467500, 2.067183, 0.314833
Epoch 6: 1.224669, 0.580600, 2.203916, 0.284167
Epoch 7: 0.850811, 0.714350, 2.738868, 0.290167
Epoch 8: 0.549040, 0.824000, 3.460863, 0.255333


wandb: uploading data; updating run metadata


Epoch 9: 0.378185, 0.881900, 3.494700, 0.282667


wandb: uploading data
wandb: uploading data; uploading wandb-summary.json; uploading config.yaml
wandb: uploading data
wandb: uploading history steps 9-9, summary
wandb: 
wandb: Run history:
wandb:          epoch ▁▂▃▃▄▅▆▆▇█
wandb:             lr ▁▁▁▁▁▁▁▁▁▁
wandb: train_accuracy ▁▂▂▃▃▄▅▆▇█
wandb:     train_loss █▇▇▇▆▅▄▃▂▁
wandb:   val_accuracy ▁▄▆███▆▇▄▆
wandb:       val_loss ▂▂▁▁▁▂▂▅██
wandb: 
wandb: Run summary:
wandb:          epoch 10
wandb:             lr 0.1
wandb: train_accuracy 0.8819
wandb:     train_loss 0.37818
wandb:   val_accuracy 0.28267
wandb:       val_loss 3.4947
wandb: 
wandb: 🚀 View run simple_model at: https://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN/runs/phrwun7j
wandb: ⭐️ View project at: https://wandb.ai/MSE_DeLearn_SPR26/MPW-CNN
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260325_111057-phrwun7j/logs
